# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n\nPublished: {meta.datePublished if hasattr(meta,'datePublished') else ''}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs. For all referencing, we use each entity's `@id` (as per Croissant best practice).


In [ ]:
# Print all record sets and their @ids
print("Available Record Sets (by @id):")
record_sets = []
for recset in getattr(meta, 'recordSet', []):
    print(f" - {recset['@id']}: {recset.get('name','(no name)')}")
    record_sets.append(recset['@id'])
# Print all fields for each record set
for recset in getattr(meta, 'recordSet', []):
    print(f"\nFields in Record Set '{recset.get('name','(no name)')}' (@id={recset['@id']}):")
    for field in recset.get('field', []):
        field_name = field.get('name', '(no name)')
        field_id = field['@id']
        field_type = field.get('dataType', '')
        print(f" • {field_name} (@id={field_id}, dataType={field_type})")
if not record_sets:
    print("No explicit record sets in Croissant metadata. Loading first distribution file as single record set.")
    if hasattr(meta, 'distribution') and len(meta.distribution) > 0:
        dist_id = meta.distribution[0]['@id']
        print(f"Using distribution @id: {dist_id}")

## 3. Data Extraction
Load data from a record set into a DataFrame for analysis. Use the record set `@id`s from the overview.

In [ ]:
# Since this dataset's Croissant schema has an empty recordSet (see metadata),
# we extract the first available table-like distribution as a record set, using its @id.
# You can inspect meta.distribution for more detail if needed.
distribution_ids = []
if hasattr(meta, 'distribution'):
    for dist in meta.distribution:
        distribution_ids.append(dist['@id'])
if not distribution_ids:
    raise ValueError("No distributions/record sets found in the dataset!")

dataframes = {}
for d_id in distribution_ids:
    try:
        records_iter = dataset.records(record_set=d_id)
        records = list(records_iter)
        dataframes[d_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for record set (@id={d_id}) with shape: {dataframes[d_id].shape}")
        print("First few columns:", dataframes[d_id].columns.tolist()[:10])
        print(dataframes[d_id].head())
    except Exception as e:
        print(f"Failed to load records for @id={d_id}: {e}\n")

# Choose the first available record set for further EDA
selected_record_set_id = distribution_ids[0]
df = dataframes[selected_record_set_id]

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All field/column accesses should use the field's `@id` as the column name wherever possible.


In [ ]:
# Let's identify numeric columns to analyze
numeric_candidates = []
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_candidates.append(col)
print("Numeric columns (via @id):", numeric_candidates)

# Choose the first numeric field (can be refined by domain knowledge)
if not numeric_candidates:
    raise ValueError("No numeric fields available for EDA.")
numeric_field_id = numeric_candidates[0]

# Filter records on that field
threshold = df[numeric_field_id].mean()  # Example cutoff: mean value
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (N={len(filtered_df)}):\n", filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a likely categorical field
str_candidates = [c for c in df.columns if pd.api.types.is_string_dtype(df[c])]
group_field_id = str_candidates[0] if str_candidates else None
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):\n", grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of chosen numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot if a grouping field is available
if group_field_id:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook we loaded, explored, and visualized the FAIR^2 mass spectrometry dataset using `mlcroissant` and referenced all columns and entities by their Croissant `@id`. Further domain-specific analyses can be designed now that the data is loaded and inspected.